# Classical Pipeline for Wound Healing Classification


**Extractors compared:** PCA, VAE, Sparse PCA, ICA, MNF

**Classifiers available:** SVM-RBF, Random Forest, Logistic Regression

## Load Libraries

In [ ]:
!pip install scanpy scikit-learn mlflow mlxtend anndata "pandas>=2.3.0"
!pip install dagshub -q
!pip install scikit-misc
!pip install torch

## Import Libraries

In [ ]:
import json
import os
import warnings

import anndata
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim
import dagshub

from scipy import sparse
from scipy.linalg import eigh

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA, SparsePCA, FastICA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, GridSearchCV, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mlxtend.plotting import plot_pca_correlation_graph

## Load Filtered Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

adata = sc.read_h5ad('/content/drive/.../final_dataset/bio_dataset.h5ad')
adata

### Remove `healing_stage` and `batch` from the dataset

Using them would lead to data leakage, since the model could use these columns as a shortcut to infer the label (0/1), making it extremely weak when applied to unseen data.

Both columns are stored separately outside the dataset so they remain available for the analysis of training/testing results.

In [ ]:
healing_stage_categories = adata.obs['healing_stage'].copy()
adata.obs.drop(columns=['healing_stage'], inplace=True)

batchs = adata.obs['batch'].copy()
adata.obs.drop(columns=['batch'], inplace=True)

adata

## Experiment Tracking with MLflow using DagsHub

To accurately assess model performance on donor genomic datasets, evaluation metrics are tracked systematically across architectural iterations. MLflow provides a centralized framework for experiment tracking, parameter logging, and model versioning, hosted via DagsHub (remote DVC + MLflow server).


In [ ]:
# ENVIRONMENT & DATA LOADING
USER_NAME = ""
REPO_NAME = ""
TOKEN = os.environ.get("DAGSHUB_TOKEN", "")

os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = TOKEN
os.environ['MLFLOW_TRACKING_URI'] = f"https://dagshub.com/{USER_NAME}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

## Group K-Fold Implementation

Group K-Fold addresses the risk of data leakage caused by genetic similarities and batch effects among cells from the same donor. By keeping all samples from a single donor entirely in either the training or the testing set, the model is forced to learn the general biological signal rather than memorizing donor-specific noise, giving a rigorous evaluation of generalization to new patients.

A nested cross-validation strategy uses two distinct loops:
- **Outer CV**: final evaluation, splitting donors into train/test to measure true generalization to unseen patients.
- **Inner CV**: runs within the training data to perform hyperparameter tuning, ensuring the test set stays unseen during model configuration.

In [ ]:
X_full = adata.X
y_full = adata.obs['healing_state'].values
donor_groups = adata.obs['donor_id'].values

outer_cv = GroupKFold(n_splits=4)

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_full, y_full, donor_groups)):
    test_donor = np.unique(donor_groups[test_idx])[0]
    train_donors = np.unique(donor_groups[train_idx])
    print(f"\n{'='*60}")
    print(f"OUTER FOLD {fold_idx + 1}/4: Testing on Donor {test_donor}")
    print(f"Training on Donors: {train_donors}")
    print(f"{'='*60}")

## Scaling Methodology

Z-score standardization ensures all genes contribute equally to the model. A custom scaling process maintains strict separation between train and test:
- **Training**: mean/std computed on the training set (zero std replaced with 1.0).
- **Testing**: the training mean/std are applied to transform the test set, so the test data is standardized according to the training distribution with no information leakage.

In [ ]:
def scale_train_test(adata_train, adata_test, max_value=10):
    sc.pp.scale(adata_train, max_value=max_value)

    train_mean = adata_train.var['mean'].values
    train_std = adata_train.var['std'].values
    train_std_safe = np.where(train_std == 0, 1.0, train_std)

    if sparse.issparse(adata_test.X):
        X_test = adata_test.X.toarray()
    else:
        X_test = np.array(adata_test.X)

    X_test_scaled = (X_test - train_mean) / train_std_safe
    X_test_scaled = np.clip(X_test_scaled, -max_value, max_value)
    adata_test.X = X_test_scaled

    adata_test.uns['scaling_info'] = {
        'mean': train_mean,
        'std': train_std_safe,
        'max_value': max_value,
        'n_zero_var_genes': int((train_std == 0).sum())
    }
    if (train_std == 0).sum() > 0:
        print(f"  \u26a0 Warning: {(train_std == 0).sum()} genes had zero variance in training set")

    return adata_train, adata_test

## Handling Class Imbalance

The dataset exhibits a severe class imbalance (~16:1 Wound:Skin). To prevent the model from ignoring the minority class:
- **Class weighting**: `class_weight='balanced'` penalizes errors on the minority class (Skin) more heavily.
- **Metric selection**: F1-score and balanced accuracy are prioritized over raw accuracy so evaluation reflects performance on both classes.

## Model Registry

In [ ]:
MODEL_REGISTRY = {
    'SVM-RBF': {
        'estimator': SVC(kernel='rbf', probability=True, random_state=42),
        'param_grid': {
            'C': [0.1, 1, 10, 100],
            'gamma': [0.001, 0.01, 0.1, 1],
            'class_weight': ['balanced']
        },
        'n_jobs': 1
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 300],
            'max_depth': [20, None],
            'class_weight': ['balanced'],
            'min_samples_split': [2, 10]
        },
        'n_jobs': 1
    },
    'LogisticRegression': {
        'estimator': LogisticRegression(random_state=42),
        'param_grid': {
            'C': np.logspace(-2, 2, 10),
            'penalty': ['l1', 'l2'],
            'solver': ['saga'],
            'class_weight': ['balanced'],
            'max_iter': [10000],
            'tol': [1e-3]
        },
        'n_jobs': 1
    }
}

## Model Selection (User Picks Model)

In [ ]:
# CHANGE THIS VARIABLE to switch models
SELECTED_MODEL = 'LogisticRegression'

if SELECTED_MODEL not in MODEL_REGISTRY:
    raise ValueError(f"Model '{SELECTED_MODEL}' not found.")

model_config = MODEL_REGISTRY[SELECTED_MODEL]
print(f"\u2713 Selected Model: {SELECTED_MODEL}")
print(f"\u2713 Configuration loaded.")

## Dimensionality Reduction

### Variational Autoencoder (VAE)

A PyTorch VAE wrapped in an sklearn-style `.fit()` / `.transform()` interface. A fresh VAE is (re)initialized for every fold to prevent leakage across folds.

In [ ]:
# --- 1. Core PyTorch VAE ---
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=50, hidden_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU()
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


# --- 2. Scikit-Learn Wrapper for VAE ---
class SklearnVAEWrapper(BaseEstimator, TransformerMixin):
    """
    Wraps PyTorch VAE to have .fit() and .transform() like sklearn PCA.
    Calling .fit() completely reinitializes and trains a new VAE.
    """
    def __init__(self, latent_dim=50, epochs=50, batch_size=64, lr=1e-3):
        self.latent_dim = latent_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.vae = None
        self.device = torch.device("cpu")

    def fit(self, X, y=None):
        torch.manual_seed(42)
        input_dim = X.shape[1]
        # Re-initialize a fresh VAE for every fold (prevents leakage)
        self.vae = VAE(input_dim=input_dim, latent_dim=self.latent_dim).to(self.device)
        optimizer = optim.Adam(self.vae.parameters(), lr=self.lr)
        X_tensor = torch.FloatTensor(X).to(self.device)

        self.vae.train()
        n_samples = X_tensor.shape[0]
        for epoch in range(self.epochs):
            permutation = torch.randperm(n_samples)
            for i in range(0, n_samples, self.batch_size):
                indices = permutation[i:i + self.batch_size]
                batch = X_tensor[indices]

                optimizer.zero_grad()
                x_recon, mu, logvar = self.vae(batch)

                recon_loss = nn.functional.mse_loss(x_recon, batch, reduction='sum')
                kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                loss = recon_loss + kl_loss

                loss.backward()
                optimizer.step()
        return self

    def transform(self, X, y=None):
        self.vae.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            mu, _ = self.vae.encode(X_tensor)
        return mu.cpu().numpy()

### Sparse PCA

In [ ]:
class SklearnSparsePCAWrapper(BaseEstimator, TransformerMixin):
    """
    Wraps SparsePCA to have .fit() and .transform() like sklearn PCA.
    Captures sparsity reports internally without printing during loops.
    """
    def __init__(self, n_components=50, alpha=1.0, max_iter=1000, n_jobs=1, random_state=42):
        self.n_components = n_components
        self.alpha = alpha
        self.max_iter = max_iter
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.sparse_pca = None
        self.sparsity_report_ = None
        self.top_genes_ = None

    def fit(self, X, feature_names=None, y=None):
        # Initialize a fresh SparsePCA for every fold (prevents leakage)
        self.sparse_pca = SparsePCA(
            n_components=self.n_components,
            alpha=self.alpha,
            max_iter=self.max_iter,
            n_jobs=self.n_jobs,
            random_state=self.random_state
        )
        print(f"Fitting SparsePCA (alpha={self.alpha})...")
        self.sparse_pca.fit(X)

        sparse_loadings = self.sparse_pca.components_
        n_nonzero = (sparse_loadings != 0).sum(axis=1)
        self.sparsity_report_ = {
            'avg_nonzero': n_nonzero.mean(),
            'min_nonzero': n_nonzero.min(),
            'max_nonzero': n_nonzero.max()
        }

        if feature_names is not None:
            self.top_genes_ = {}
            for comp_idx in range(min(5, self.n_components)):
                nonzero_idx = np.nonzero(sparse_loadings[comp_idx])[0]
                if len(nonzero_idx) > 0:
                    self.top_genes_[f"SPC{comp_idx+1}"] = list(feature_names[nonzero_idx])
        return self

    def transform(self, X, y=None):
        return self.sparse_pca.transform(X)

### Independent Component Analysis (ICA)

Decomposes high-dimensional data into a set of statistically independent components.

In [ ]:
class SklearnICAWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=50, max_iter=1000, random_state=42, whiten='unit-variance'):
        self.n_components = n_components
        self.max_iter = max_iter
        self.random_state = random_state
        self.whiten = whiten
        self.ica = None

    def fit(self, X, y=None):
        self.ica = FastICA(
            n_components=self.n_components,
            max_iter=self.max_iter,
            random_state=self.random_state,
            whiten=self.whiten
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.ica.fit(X)
        return self

    def transform(self, X, y=None):
        return self.ica.transform(X)

### Minimum/Maximum Noise Fraction (MNF)

Maximizes the signal-to-noise ratio of the data via a noise-whitening step followed by PCA on the whitened data. Components with the highest signal quality are retained; noisy components are discarded.

In [ ]:
class MNFWrapper(BaseEstimator, TransformerMixin):
    """
    Maximum Noise Fraction. Orders components by signal-to-noise ratio.
    Noise covariance is estimated via nearest-neighbor differencing, WITHIN the data
    passed to .fit() only.
    """
    def __init__(self, n_components=50, k_neighbors=5, reg_eps=1e-6, random_state=42):
        self.n_components = n_components
        self.k_neighbors = k_neighbors
        self.reg_eps = reg_eps
        self.random_state = random_state
        self.mean_ = None
        self.components_ = None
        self.snr_ = None

    def _estimate_noise_cov(self, X):
        nn = NearestNeighbors(n_neighbors=self.k_neighbors + 1).fit(X)
        _, idx = nn.kneighbors(X)
        neighbor_idx = idx[:, 1]
        diffs = X - X[neighbor_idx]
        return np.cov(diffs, rowvar=False) / 2.0

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        signal_cov = np.cov(Xc, rowvar=False)
        noise_cov = self._estimate_noise_cov(X)
        reg = self.reg_eps * np.trace(noise_cov) / noise_cov.shape[0]
        noise_cov += reg * np.eye(noise_cov.shape[0])

        eigvals, eigvecs = eigh(signal_cov, noise_cov)
        order = np.argsort(eigvals)[::-1]
        self.components_ = eigvecs[:, order[:self.n_components]].T
        self.snr_ = eigvals[order[:self.n_components]]
        return self

    def transform(self, X, y=None):
        Xc = np.asarray(X) - self.mean_
        return Xc @ self.components_.T

### Sparse-PCA Alpha Selection

A quick tuning pass on a 1,000-cell subset to pick a SparsePCA `alpha` that yields roughly 50–200 genes per component (run once, not inside the CV loop).

In [ ]:
# 1. Force subset to 1000 cells
adata_subset = adata[:1000].copy()

# 2. Force normalization
sc.pp.normalize_total(adata_subset, target_sum=1e4)
sc.pp.log1p(adata_subset)

# 3. Force HVG selection
sc.pp.highly_variable_genes(adata_subset, flavor='seurat', n_top_genes=2000)
adata_subset = adata_subset[:, adata_subset.var['highly_variable']]

# 4. Force dense array
X_tuning = adata_subset.X.toarray() if hasattr(adata_subset.X, 'toarray') else np.array(adata_subset.X)

# 5. Safety check (prevents multi-minute freezes on the full gene set)
assert X_tuning.shape[1] == 2000, f"ERROR: Still have {X_tuning.shape[1]} genes! HVG step failed."
print(f"Success! Shape is {X_tuning.shape}. Testing alphas...")

# 6. Fast tuning (max_iter=50 is enough to see the trend)
for alpha_test in [1.0, 2.0, 5.0, 10.0]:
    print(f"alpha={alpha_test}...", end=" ", flush=True)
    spca = SparsePCA(n_components=10, alpha=alpha_test, max_iter=50, n_jobs=1, random_state=42)
    spca.fit(X_tuning)
    avg_genes = (spca.components_ != 0).sum(axis=1).mean()
    print(f"-> {avg_genes:.0f} genes/comp")

print("Done! Pick the alpha that gives ~50-200 genes.")

### Feature Extractor Registry

In [ ]:
FEATURE_EXTRACTOR_REGISTRY = {
    'PCA': {
        'extractor': PCA(n_components=50, random_state=42)
    },
    'VAE': {
        'extractor': SklearnVAEWrapper(latent_dim=50, epochs=50)
    },
    'SparsePCA': {
        'extractor': SklearnSparsePCAWrapper(n_components=50, alpha=5.0, n_jobs=1)
    },
    'ICA': {
        'extractor': SklearnICAWrapper(n_components=50, random_state=42)
    },
    'MNF': {
        'extractor': MNFWrapper(n_components=50, k_neighbors=5)
    },
}

## Pipeline in a Loop, with Data-Leakage Avoidance

Pick a feature extractor via `SELECTED_EXTRACTOR`, then rerun this section. The outer loop below performs preprocessing, feature extraction, hyperparameter tuning, evaluation and MLflow logging per donor fold.

In [ ]:
os.environ["SCIPY_ARRAY_API"] = "1"

import copy

# 1. Select Feature Extractor
SELECTED_EXTRACTOR = 'ICA'  # Change to 'PCA', 'VAE', 'SparsePCA', 'ICA', or 'MNF' and rerun

if SELECTED_EXTRACTOR not in FEATURE_EXTRACTOR_REGISTRY:
    raise ValueError(f"Extractor '{SELECTED_EXTRACTOR}' not found.")

feature_extractor = copy.deepcopy(FEATURE_EXTRACTOR_REGISTRY[SELECTED_EXTRACTOR]['extractor'])

In [ ]:
# ==========================================
# GENE IMPORTANCE EXTRACTION (for GO enrichment)
# ==========================================
def extract_top_genes(extractor_name, feature_extractor, hvg_genes, n_top=10):
    """
    Sum of squared component loadings -> per-gene importance -> top-N genes.
    Weighted by explained variance / SNR where available (PCA, MNF);
    unweighted for ICA/SparsePCA, whose components are not variance-ordered.
    VAE is not linear-decomposable and is not handled here.
    """
    hvg_arr = np.array(hvg_genes)

    if extractor_name == 'PCA':
        loadings = feature_extractor.components_
        weights = feature_extractor.explained_variance_ratio_
        importance = np.sum((loadings ** 2) * weights[:, None], axis=0)
    elif extractor_name == 'SparsePCA':
        loadings = feature_extractor.sparse_pca.components_
        importance = np.sum(loadings ** 2, axis=0)
    elif extractor_name == 'ICA':
        loadings = feature_extractor.ica.components_
        importance = np.sum(loadings ** 2, axis=0)
    elif extractor_name == 'MNF':
        loadings = feature_extractor.components_
        weights = feature_extractor.snr_
        importance = np.sum((loadings ** 2) * weights[:, None], axis=0)
    else:
        return None, None  # VAE

    top_idx = np.argsort(importance)[::-1][:n_top]
    return hvg_arr[top_idx].tolist(), importance[top_idx].tolist()


mlflow.set_experiment("Wound_Healing_Classical_Path_A")

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_full, y_full, groups=donor_groups)):
    test_donor = np.unique(donor_groups[test_idx])[0]
    train_donors = np.unique(donor_groups[train_idx])
    print(f"\nFold {fold_idx+1}: Test on {test_donor} | Train on {train_donors}")

    # --- PREPROCESSING ---
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()

    sc.pp.normalize_total(adata_train, target_sum=1e4)
    sc.pp.log1p(adata_train)
    sc.pp.normalize_total(adata_test, target_sum=1e4)
    sc.pp.log1p(adata_test)

    sc.pp.highly_variable_genes(adata_train, flavor='seurat', n_top_genes=2000)
    hvg_genes = adata_train.var[adata_train.var['highly_variable']].index
    adata_train = adata_train[:, hvg_genes]
    adata_test = adata_test[:, hvg_genes]

    adata_train, adata_test = scale_train_test(adata_train, adata_test, max_value=10)

    X_train_dense = adata_train.X if not sparse.issparse(adata_train.X) else adata_train.X.toarray()
    X_test_dense = adata_test.X if not sparse.issparse(adata_test.X) else adata_test.X.toarray()

    # --- DYNAMIC FEATURE EXTRACTION ---
    print(f"Extracting features using: {SELECTED_EXTRACTOR}...")
    if SELECTED_EXTRACTOR == 'SparsePCA':
        x_train = feature_extractor.fit_transform(X_train_dense, feature_names=hvg_genes)
    else:
        x_train = feature_extractor.fit_transform(X_train_dense)
    x_test = feature_extractor.transform(X_test_dense)

    adata_train.obsm[f"X_{SELECTED_EXTRACTOR.lower()}"] = x_train
    adata_test.obsm[f"X_{SELECTED_EXTRACTOR.lower()}"] = x_test

    if SELECTED_EXTRACTOR == 'SparsePCA' and hasattr(feature_extractor, 'sparsity_report_'):
        print(f"  -> Sparsity: Avg {feature_extractor.sparsity_report_['avg_nonzero']:.1f} genes/comp")

    # --- TOP-10 GENE EXTRACTION (for GO enrichment) ---
    top_genes, top_gene_scores = extract_top_genes(SELECTED_EXTRACTOR, feature_extractor, hvg_genes, n_top=10)
    if top_genes is not None:
        print(f"  -> Top 10 genes: {top_genes}")

    y_train = adata_train.obs['healing_state'].values
    y_test = adata_test.obs['healing_state'].values
    if x_train.shape[0] != y_train.shape[0]:
        break

    # --- MODEL TRAINING ---
    inner_cv = GroupKFold(n_splits=3)
    inner_groups = adata_train.obs['donor_id'].values

    grid = GridSearchCV(
        estimator=model_config['estimator'],
        param_grid=model_config['param_grid'],
        cv=inner_cv.split(x_train, y_train, groups=inner_groups),
        scoring='f1',
        n_jobs=1,
        verbose=1
    )
    print(f"Training {SELECTED_MODEL}...")
    grid.fit(x_train, y_train)
    best_model = grid.best_estimator_

    y_pred = best_model.predict(x_test)
    y_proba = best_model.predict_proba(x_test)[:, 1]
    y_train_pred = grid.best_estimator_.predict(x_train)

    train_f1 = f1_score(y_train, y_train_pred)
    f1 = f1_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # --- MLFLOW LOGGING ---
    with mlflow.start_run(run_name=f"{SELECTED_EXTRACTOR}_{SELECTED_MODEL}_Donor_{test_donor}"):
        mlflow.log_param("feature_extractor", SELECTED_EXTRACTOR)
        mlflow.log_param("classifier", SELECTED_MODEL)
        mlflow.log_param("test_donor", str(test_donor))
        mlflow.log_param("n_train_samples", len(x_train))
        mlflow.log_param("n_test_samples", len(x_test))
        mlflow.log_param("n_components", 50)
        for param, value in grid.best_params_.items():
            mlflow.log_param(f"classifier_{param}", value)

        mlflow.log_metric("cv_f1_score", grid.best_score_)
        mlflow.log_metric("train_f1_score", train_f1)
        mlflow.log_metric("test_f1_score", f1)
        mlflow.log_metric("test_balanced_accuracy", bal_acc)
        mlflow.log_metric("test_roc_auc", roc)
        mlflow.log_metric("true_negatives", int(tn))
        mlflow.log_metric("false_positives", int(fp))
        mlflow.log_metric("false_negatives", int(fn))
        mlflow.log_metric("true_positives", int(tp))

        mlflow.sklearn.log_model(best_model, f"{SELECTED_MODEL.lower()}_model")

        extractor_filename = f'{SELECTED_EXTRACTOR.lower()}_extractor_{test_donor}.joblib'
        joblib.dump(feature_extractor, extractor_filename)
        mlflow.log_artifact(extractor_filename)

        hvg_filename = f'hvg_genes_{SELECTED_EXTRACTOR.lower()}_{test_donor}.json'
        with open(hvg_filename, 'w') as f:
            json.dump(list(hvg_genes), f, indent=2)
        mlflow.log_artifact(hvg_filename)

        if top_genes is not None:
            top_genes_filename = f'top_genes_{SELECTED_EXTRACTOR.lower()}_{test_donor}.json'
            with open(top_genes_filename, 'w') as f:
                json.dump({'top_genes': top_genes, 'scores': top_gene_scores}, f, indent=2)
            mlflow.log_artifact(top_genes_filename)

        results = {
            'path': f'Path A ({SELECTED_EXTRACTOR})',
            'feature_extractor': SELECTED_EXTRACTOR,
            'classifier': SELECTED_MODEL,
            'test_donor': test_donor,
            'cv_f1': float(grid.best_score_),
            'test_f1': float(f1),
            'test_balanced_acc': float(bal_acc),
            'test_roc_auc': float(roc),
            'train_f1': float(train_f1),
            'best_params': grid.best_params_,
            'confusion_matrix': cm.tolist(),
            'top_genes': top_genes,
            'top_gene_scores': top_gene_scores
        }
        filename = f'{SELECTED_EXTRACTOR.lower()}_{SELECTED_MODEL.lower()}_{test_donor}.json'
        with open(filename, 'w') as f:
            json.dump(results, f, indent=2)
        mlflow.log_artifact(filename)

        print(f"\u2713 Fold {test_donor} logged to MLflow.")